# Temporary: Add Westervoort to Discharge Stations

Goal: Add Westervoort station (IJssel entrance) to the discharge_stations layer in water_stations_for_modeling.gpkg, matching the structure of Millingen a/d Rijn (Waal entrance).

In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from pathlib import Path

# Path to the geopackage
GPKG_PATH = Path("../../data/archive_water/water_stations_by_parameter/water_stations_for_modeling.gpkg")

print(f"📂 Loading: {GPKG_PATH}")
print(f"   Exists: {GPKG_PATH.exists()}\n")

📂 Loading: ../../data/archive_water/water_stations_by_parameter/water_stations_for_modeling.gpkg
   Exists: True



In [3]:
# List layers (using geopandas instead of fiona)
import subprocess
result = subprocess.run(['ogrinfo', str(GPKG_PATH)], capture_output=True, text=True)
if result.returncode == 0:
    # Parse layer names from ogrinfo output
    lines = result.stdout.split('\n')
    layers = [line.split(':')[1].strip() for line in lines if line.startswith('1:') or line.startswith('2:') or line.startswith('3:')]
    if not layers:
        # Alternative: just try to read and catch errors
        layers = []
        for potential_layer in ['discharge_stations', 'water_height_stations', 'discharge_calculation_metadata']:
            try:
                gpd.read_file(GPKG_PATH, layer=potential_layer, rows=0)
                layers.append(potential_layer)
            except:
                pass
    print(f"📋 Layers in geopackage: {layers}\n")
else:
    print("📋 Will check layers by attempting to read them...\n")
    layers = ['discharge_stations', 'water_height_stations']

📋 Layers in geopackage: ['water_height_stations (Point)', 'discharge_stations (Point)', 'discharge_calculation_metadata (Point)']



In [5]:
# Read discharge_stations layer
discharge_gdf = gpd.read_file(GPKG_PATH, layer='discharge_stations')
print(f"✅ Loaded discharge_stations: {len(discharge_gdf)} stations\n")
print("Columns:")
print(discharge_gdf.columns.tolist())
print(f"\nDataFrame info:")
print(discharge_gdf.info())
print("\nFirst few rows:")
print(discharge_gdf.head())

# Try to find station code column (could be 'CODE', 'code', 'station_code', etc.)
code_col = None
name_col = None
for col in discharge_gdf.columns:
    col_lower = col.lower()
    if 'code' in col_lower and code_col is None:
        code_col = col
    if 'name' in col_lower or 'naam' in col_lower:
        name_col = col

print(f"\n🔍 Detected columns:")
print(f"   Code column: {code_col}")
print(f"   Name column: {name_col}")

if code_col:
    print(f"\n📍 Check if Westervoort or Millingen exist:")
    mask = discharge_gdf[code_col].astype(str).str.contains('wester|milling', case=False, na=False)
    if name_col:
        print(discharge_gdf[mask][[code_col, name_col]])
    else:
        print(discharge_gdf[mask])

✅ Loaded discharge_stations: 13 stations

Columns:
['CODE', 'NAAM', 'PARAMETER_WAT_OMSCHRIJVING', 'TIJDSTIP_LAATSTE_METING', 'geometry']

DataFrame info:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 5 columns):
 #   Column                      Non-Null Count  Dtype              
---  ------                      --------------  -----              
 0   CODE                        13 non-null     object             
 1   NAAM                        13 non-null     object             
 2   PARAMETER_WAT_OMSCHRIJVING  13 non-null     object             
 3   TIJDSTIP_LAATSTE_METING     13 non-null     datetime64[ms, UTC]
 4   geometry                    13 non-null     geometry           
dtypes: datetime64[ms, UTC](1), geometry(1), object(3)
memory usage: 652.0+ bytes
None

First few rows:
                                CODE                                  NAAM  \
0  maastricht.borgharen.maas.beneden  Maastricht, Borgharen, Maas, bene

In [6]:
# Get Millingen structure as template
if code_col:
    millingen = discharge_gdf[discharge_gdf[code_col].astype(str).str.contains('milling', case=False, na=False)]
    if len(millingen) > 0:
        print("📋 Millingen structure (template):")
        print(millingen.iloc[0])
    else:
        print("⚠️ No Millingen found. Showing first row as template:")
        print(discharge_gdf.iloc[0])
else:
    print("⚠️ No code column found. Showing first row:")
    print(discharge_gdf.iloc[0])

📋 Millingen structure (template):
CODE                                                    millingenaanderijn
NAAM                                                    Millingen a/d Rijn
PARAMETER_WAT_OMSCHRIJVING              Debiet in Oppervlaktewater in m3/s
TIJDSTIP_LAATSTE_METING                          2026-02-13 21:10:00+00:00
geometry                      POINT (199599.99257660407 431800.0956836164)
Name: 12, dtype: object


In [7]:
# Check if westervoort already exists
if code_col:
    westervoort_exists = discharge_gdf[code_col].astype(str).str.contains('westervoort', case=False, na=False).any()
    print(f"Westervoort exists in discharge_stations: {westervoort_exists}")
    
    if westervoort_exists:
        print("\n✅ Westervoort already in discharge_stations:")
        mask = discharge_gdf[code_col].astype(str).str.contains('westervoort', case=False, na=False)
        print(discharge_gdf[mask])
    else:
        print("\n⚠️ Westervoort NOT in discharge_stations. Will add it.")
else:
    print("⚠️ Cannot check - no code column found")
    westervoort_exists = False

Westervoort exists in discharge_stations: True

✅ Westervoort already in discharge_stations:
                     CODE                   NAAM  \
11  westervoort.ijsselkop  Westervoort IJsselkop   

            PARAMETER_WAT_OMSCHRIJVING   TIJDSTIP_LAATSTE_METING  \
11  Debiet in Oppervlaktewater in m3/s 2024-06-18 01:50:00+00:00   

                         geometry  
11  POINT (193896.873 440402.257)  


In [9]:
# Westervoort coordinates (RD New / EPSG:28992)
# Try to get coordinates from water_height_stations layer
try:
    water_height_gdf = gpd.read_file(GPKG_PATH, layer='water_height_stations')
    
    # Find code column in water height stations
    wh_code_col = None
    for col in water_height_gdf.columns:
        if 'code' in col.lower():
            wh_code_col = col
            break
    
    if wh_code_col:
        westervoort_wh = water_height_gdf[water_height_gdf[wh_code_col].astype(str).str.contains('westervoort', case=False, na=False)]
        if len(westervoort_wh) > 0:
            print("📍 Found Westervoort in water_height_stations:")
            print(westervoort_wh[[wh_code_col, 'geometry']].head())
            # Use the first one (likely westervoort.ijsselkop)
            westervoort_geom = westervoort_wh.iloc[0].geometry
            westervoort_x = westervoort_geom.x
            westervoort_y = westervoort_geom.y
            print(f"\n✅ Using coordinates: X={westervoort_x:.2f}, Y={westervoort_y:.2f}")
        else:
            print("⚠️ No Westervoort in water_height_stations, using default coordinates")
            westervoort_x = 195000
            westervoort_y = 435000
    else:
        print("⚠️ No code column in water_height_stations, using default coordinates")
        westervoort_x = 195000
        westervoort_y = 435000
except Exception as e:
    print(f"⚠️ Could not read water_height_stations: {e}")
    print("Using default coordinates for Westervoort")
    westervoort_x = 195000
    westervoort_y = 435000

📍 Found Westervoort in water_height_stations:
                                      CODE                       geometry
23                   westervoort.ijsselkop  POINT (193896.873 440402.257)
43                           westervoort.2  POINT (194498.153 442609.955)
62    westervoort.hondsbroekschepleij.geul  POINT (193903.039 441005.007)
65  westervoort.hondsbroekschepleij.ijssel  POINT (193699.102 441427.995)

✅ Using coordinates: X=193896.87, Y=440402.26


In [ ]:
# Create Westervoort entry (matching existing structure)
if not westervoort_exists and code_col:
    # Build new row matching the actual columns in the geopackage
    new_row_data = {
        'geometry': Point(westervoort_x, westervoort_y)
    }
    
    # Add code and name using detected column names
    if code_col:
        new_row_data[code_col] = 'westervoort'
    if name_col:
        new_row_data[name_col] = 'Westervoort (IJssel entrance)'
    
    # Add other common fields if they exist
    for col in discharge_gdf.columns:
        if col == 'geometry' or col in new_row_data:
            continue
        col_lower = col.lower()
        
        # Try to fill in reasonable values based on column name
        if 'parameter' in col_lower and 'code' not in col_lower:
            new_row_data[col] = 'Debiet (calculated)'
        elif 'parameter' in col_lower and 'code' in col_lower:
            new_row_data[col] = 'Q'
        elif 'year' in col_lower:
            new_row_data[col] = 9  # 2016-2024
        elif 'calculated' in col_lower or 'calc' in col_lower:
            new_row_data[col] = True
        elif 'method' in col_lower or 'calculation' in col_lower:
            new_row_data[col] = 'Q_ijssel = Q_pannerden - Q_driel (mass balance)'
        else:
            # Get dtype and set None or appropriate default
            if col in discharge_gdf.columns:
                new_row_data[col] = None
    
    # Create new GeoDataFrame with the same CRS
    new_station = gpd.GeoDataFrame([new_row_data], crs=discharge_gdf.crs)
    
    # Append to existing
    discharge_gdf_updated = pd.concat([discharge_gdf, new_station], ignore_index=True)
    
    print("✅ Created new Westervoort entry:")
    print(new_station)
    print(f"\n📊 Updated discharge_stations: {len(discharge_gdf_updated)} stations (was {len(discharge_gdf)})")
else:
    discharge_gdf_updated = discharge_gdf
    if westervoort_exists:
        print("No changes needed - Westervoort already exists")
    else:
        print("⚠️ Cannot add - missing required columns")

In [ ]:
# Save back to geopackage
if not westervoort_exists:
    # Create backup first
    import shutil
    backup_path = GPKG_PATH.parent / f"{GPKG_PATH.stem}_backup{GPKG_PATH.suffix}"
    shutil.copy2(GPKG_PATH, backup_path)
    print(f"💾 Backup created: {backup_path}")
    
    # Write updated layer
    discharge_gdf_updated.to_file(GPKG_PATH, layer='discharge_stations', driver='GPKG')
    print(f"✅ Updated discharge_stations layer in {GPKG_PATH}")
    
    # Verify
    verify_gdf = gpd.read_file(GPKG_PATH, layer='discharge_stations')
    westervoort_verify = verify_gdf[verify_gdf['station_code'] == 'westervoort']
    if len(westervoort_verify) > 0:
        print("\n✅ VERIFICATION SUCCESSFUL:")
        print(westervoort_verify[['station_code', 'station_name', 'is_calculated', 'geometry']])
    else:
        print("\n❌ VERIFICATION FAILED: Westervoort not found after save")
else:
    print("No save needed - no changes made")

In [ ]:
# Summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
final_gdf = gpd.read_file(GPKG_PATH, layer='discharge_stations')
print(f"\n📊 Total discharge stations: {len(final_gdf)}")

# Find code column again
final_code_col = None
for col in final_gdf.columns:
    if 'code' in col.lower():
        final_code_col = col
        break

if final_code_col:
    print(f"\n🔍 All station codes:")
    print(final_gdf[final_code_col].tolist())
    
    # Look for calculated stations
    print(f"\n🧮 Checking for Westervoort and Millingen:")
    mask = final_gdf[final_code_col].astype(str).str.contains('wester|milling', case=False, na=False)
    print(final_gdf[mask][[final_code_col] + [c for c in final_gdf.columns if c != final_code_col and c != 'geometry']])